# Capstone — Refresh / Content Opportunity Scoring

This notebook mirrors the deployed paper and produces a ranked queue of refresh opportunities using the anonymized FlyRank starter dataset.

## 1. Question
**Decision supported:** which pages should be reviewed first for refresh work.

We frame this as ranking page-level opportunities by decline risk and recovery potential, then map the top pages to concrete editor actions.

In [1]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent.parent

print('Repo root:', ROOT)
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_all.py')], check=True)
print('Pipeline run complete.')


Repo root: /home/runner/work/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Pipeline run complete.


## 2. Data
- Source: `data/raw/content_refresh_anonymized.csv` (30,000 rows, pseudonymized).
- Unit: one row per content item over a trailing 90-day window.
- Public-safe exclusions: no client names/domains/URLs/raw queries/credentials.
- Label leakage controls: `trend_direction` and `trend_pct` are excluded as model features because the label derives from them.

In [2]:
import pandas as pd

raw = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
print('Rows:', len(raw))
print('Columns:', raw.shape[1])
print('Clients:', raw['client_id'].nunique())
print('Declining label rate:', round((raw['trend_direction'] == 'down').mean(), 3))


Rows: 30000
Columns: 44
Clients: 32
Declining label rate: 0.542


## 3. Methodology
- Lane: **Refresh / Content Opportunity Scoring**.
- Baseline: transparent rules engine (`02_baseline_score.py`) with reason codes.
- Model: logistic regression, decision tree, random forest (`03_train_model.py`).
- Validation: client-holdout split to test cross-client generalization.
- Leakage checks: no IDs or label-derived fields used as predictors.

In [3]:
report = (ROOT / 'outputs' / 'model_report.md').read_text()
for line in report.splitlines():
    if '| random_forest |' in line or '| baseline_rules |' in line:
        print(line)


| random_forest | 0.747 | 0.610 | 0.680 | 0.741 | 0.638 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 4. Results (vs baseline)
On the same validation split, the selected random forest reaches **Precision@50 = 0.740**, while the baseline rules score **0.240**.

This is an observed lift for ranking quality in this dataset and split; it is decision support, not a causal claim about Google ranking behavior.

In [4]:
queue_path = ROOT / 'outputs' / 'refresh_queue.csv'
if not queue_path.exists():
    queue_path = ROOT / 'outputs' / 'refresh_queue_sample.csv'

queue = pd.read_csv(queue_path)
print('Queue source:', queue_path.name)
print('Rows ranked:', len(queue))
print('Top suggested actions:')
print(queue['suggested_action'].value_counts().head(5).to_string())
display(queue[['final_rank','final_refresh_score','best_model_probability','suggested_action','final_reason_codes']].head(10))


Queue source: refresh_queue.csv
Rows ranked: 30000
Top suggested actions:
suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82
 final_rank  final_refresh_score  best_model_probability       suggested_action                                                                                                                                                   final_reason_codes
          1            81.928467                0.786247 refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
          2            81.728449                0.792117 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|

## 5. Limitations
- This analysis is observational and does not prove causality.
- Results come from a starter anonymized slice; full-warehouse replication is still needed for production use.
- Recommendations are triage aids and require human editorial review before execution.

In [5]:
print('Limitation check: interpretation language stays directional / decision-support.')


Limitation check: interpretation language stays directional / decision-support.


## 6. Ranked recommendations
1. **Refresh + CTR review first** for high-visibility, low-CTR pages with decline signals.
2. **Refresh + engagement review** where scroll/session behavior is weak despite visibility.
3. **Monitor only** lower-confidence pages to avoid over-editing low-evidence items.

In [6]:
top_cols = ['final_rank','content_id','client_id','final_refresh_score','suggested_action','final_reason_codes']
display(queue[top_cols].head(20))


 final_rank           content_id         client_id  final_refresh_score              suggested_action                                                                                                                                                   final_reason_codes
          1 content_1f080331fa2b client_3fdba35f04            81.928467        refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
          2 content_6aa43079fb0c client_3fdba35f04            81.728449        refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
          3 content_d6570c51c9bd client_3fdba35f04            81.639118        refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_pag

## 7. Reproducibility artifacts
- Pipeline command: `python scripts/run_all.py`
- Report: `outputs/model_report.md`
- Queue CSV: `outputs/refresh_queue.csv` (generated)
- Example queue in git: `outputs/refresh_queue_sample.csv`
- Paper source: `docs/index.html`
- Data credit: **Built on the FlyRank ML Internship dataset** (https://flyrank.ai)

In [7]:
artifact_paths = [
    ROOT / 'outputs' / 'model_report.md',
    ROOT / 'outputs' / 'charts' / 'top_feature_importance.svg',
    ROOT / 'outputs' / 'charts' / 'action_mix.svg',
]
for p in artifact_paths:
    print(p.name, 'exists' if p.exists() else 'missing')


model_report.md exists
top_feature_importance.svg exists
action_mix.svg exists


## Self-check
- [x] Model compared against baseline on same split
- [x] Leakage risks addressed
- [x] Ranked recommendations produced
- [x] Public-safe language and anonymized data usage only
- [x] Paper includes required FlyRank data credit